# **Laboratory of Wearable Biosignal Processing**

# **02 - Biosignals-applied Machine Learning: Preprocessing and Training Pipeline**

## Before You Start

```bash
git pull
```


## Dataset: PTB-XL (Subset)

Same dataset as session 01: subset of the [PTB-XL](https://physionet.org/content/ptb-xl/1.0.3/) dataset (first 100 subjects, 12-lead ECG at 100/500 Hz).


## Data Loading and Quick Visualization

Load metadata and all ECG records, pick a reproducible random record (fixed seed), and plot all 12 leads as a sanity check before entering the preprocessing pipeline.


In [ ]:
# Imports for data loading, handling, plotting, filtering, and ECG peak analysis
import ast
import os
import wfdb

import matplotlib.pyplot as plt
import neurokit2 as nk
import numpy as np
import pandas as pd

from scipy import signal

In [ ]:
# Root folder containing PTB-XL metadata and waveform files.
DATA_DIR = "../data"
# Fixed seed to keep random record selection reproducible across runs.
SEED = 42

# Output folder for all generated figures (grouped by notebook context).
NOTEBOOK_STEM = "02_biosignals_preprocessing_pipeline"
NOTEBOOK_CONTEXT = os.path.basename(os.getcwd())

OUTPUTS_DIR = os.path.abspath(os.path.join("..", "outputs", f"{NOTEBOOK_CONTEXT}_{NOTEBOOK_STEM}"))
if not os.path.exists(OUTPUTS_DIR):
    os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Global counter to enforce ordered names: figure_01_..., figure_02_..., etc.
FIGURE_COUNTER = 1


In [ ]:
def save_figure(fig, suffix):
    """Save a matplotlib figure using the required sequential naming convention."""
    global FIGURE_COUNTER
    filename = f"figure_{FIGURE_COUNTER:02d}_{suffix}.png"
    output_path = os.path.join(OUTPUTS_DIR, filename)
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    print(f"Saved figure: {output_path}")
    FIGURE_COUNTER += 1

def get_record_filename(row, sampling_rate):
    """
    ## Description
    Choose the PTB-XL record path according to the desired sampling rate.

    ## Parameters
    - `row`: a row of the PTB-XL metadata dataframe containing the record information.
    - `sampling_rate`: the desired sampling rate (100 or 500 Hz) for loading the record.
    """

    # PTB-XL stores low-rate and high-rate paths in separate metadata columns
    # This function is an helper to be used to extract the correct path for loading a record at the desired sampling rate
    if sampling_rate == 100:
        return row["filename_lr"]
    if sampling_rate == 500:
        return row["filename_hr"]
    raise ValueError("sampling_rate must be 100 or 500")


def load_ecg_record(row, sampling_rate, data_dir):
    """
    ## Description
    Load one ECG record (WFDB) from PTB-XL metadata row.

    ## Parameters
    - `row`: a row of the PTB-XL metadata dataframe containing the record information.
    - `sampling_rate`: the desired sampling rate (100 or 500 Hz) for loading the record.
    - `data_dir`: the root folder containing PTB-XL metadata and waveform files.
    """

    # Extract the path to the record file according to the desired sampling rate
    rel_path = get_record_filename(row, sampling_rate)

    # WFDB expects a path without extension (it resolves .hea/.dat files).
    full_path = f"{data_dir}/{rel_path}"

    return wfdb.rdrecord(full_path)  # Return the loaded record as a WFDB Record object


def load_all_ecg_records(metadata_df, sampling_rate, data_dir):
    """
    ## Description
    Load all ECG records at a given sampling rate and return a dict keyed by ecg_id.

    ## Parameters
    - `metadata_df`: the PTB-XL metadata dataframe containing information about all records.
    - `sampling_rate`: the desired sampling rate (100 or 500 Hz) for loading the records.
    - `data_dir`: the root folder containing PTB-XL metadata and waveform files.
    """
    records_by_ecg_id = {}

    # Iterate metadata rows and cache each loaded record for quick random access.
    for _, row in metadata_df.iterrows():  # This helps in extracting the index and the row content at the same time
        # Take the ecg_id as the key for the loaded record
        ecg_id = int(row["ecg_id"])

        # Load the record using the helper function and store it in the dict as a wfdb Record object
        records_by_ecg_id[ecg_id] = load_ecg_record(row, sampling_rate, data_dir)

    return records_by_ecg_id


def plot_12_leads_two_columns(
    record,
    sampling_rate,
    patient_id,
    ecg_id,
    max_seconds=10.0,
    apply_standard_normalization=False,
):
    """
    ## Description
    Plot 12 ECG leads with 6 leads in the left column and 6 in the right column.

    ## Parameters
    - `record`: a WFDB Record object containing the ECG signal and metadata to plot.
    - `sampling_rate`: the sampling rate of the ECG signal (100 or 500 Hz for PTB-XL).
    - `patient_id`: the patient ID to display in the plot title.
    - `ecg_id`: the ECG ID to display in the plot title.
    - `max_seconds`: the maximum number of seconds to plot (must be <= 10.0).
    - `apply_standard_normalization`: whether to apply standard normalization (z-score) to each lead independently over the plotted window (default: False).
    """

    if max_seconds > 10.0:
        raise ValueError("max_seconds cannot be greater than 10.0")
    if max_seconds <= 0:
        raise ValueError("max_seconds must be greater than 0")

    # Paper-ready font sizes
    title_fs = 13
    label_fs = 12
    tick_fs = 11
    suptitle_fs = 17

    # Extract the signal and lead names from the WFDB Record object
    signal = record.p_signal.copy()
    lead_names = record.sig_name

    # Restrict the visible window to improve readability on long recordings.
    n_samples = min(signal.shape[0], int(max_seconds * sampling_rate))
    t = np.arange(n_samples) / sampling_rate

    if apply_standard_normalization:
        # Normalize each lead independently over the plotted window.
        means = np.mean(signal[:n_samples, :], axis=0)
        stds = np.std(signal[:n_samples, :], axis=0)
        stds[stds == 0] = 1.0
        signal[:n_samples, :] = (signal[:n_samples, :] - means) / stds

    # Share y-axis only when normalized; otherwise keep independent scales.
    fig, axes = plt.subplots(
        6,
        2,
        figsize=(16, 16),
        sharex=True,
        sharey=apply_standard_normalization,
    )

    # Place leads 0..5 in the left column and 6..11 in the right column.
    for lead_idx in range(12):
        col = 0 if lead_idx < 6 else 1
        row = lead_idx if lead_idx < 6 else lead_idx - 6

        ax = axes[row, col]
        ax.plot(t, signal[:n_samples, lead_idx], linewidth=1.0)
        ax.set_title(lead_names[lead_idx], fontsize=title_fs)
        ax.set_ylabel("z-score" if apply_standard_normalization else "mV", fontsize=label_fs)
        ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)
        ax.grid(True, alpha=0.7)

    axes[5, 0].set_xlabel("Time [s]", fontsize=label_fs)
    axes[5, 1].set_xlabel("Time [s]", fontsize=label_fs)
    axes[0, 0].set_xlim(0, max_seconds)

    norm_label = "normalized" if apply_standard_normalization else "raw"
    fig.suptitle(
        f"PTB-XL ECG | patient_id={patient_id} | ecg_id={ecg_id} | fs={sampling_rate} Hz | {norm_label}",
        fontsize=suptitle_fs,
    )
    plt.tight_layout()
    save_figure(fig, "12_leads")
    plt.show()

In [ ]:
# Load the entire metadata table (one row per ECG exam).
metadata_path = f"{DATA_DIR}/ptbxl_database.csv"
ptbxl_df = pd.read_csv(metadata_path)

print(f"Metadata shape: {ptbxl_df.shape}")
print("First columns:", list(ptbxl_df.columns[:10]))
display(ptbxl_df.head(3))

In [ ]:
# Choose here: 100 for low-resolution or 500 for high-resolution.
sampling_rate_to_plot = 500

# Choose here: True to standardize each lead (z-score), False to keep raw amplitudes.
apply_standard_normalization = False

# Choose here: how many seconds of ECG to visualize in the final plot (max 10.0).
max_seconds_to_plot = 10.0

# Load all ECG signals once for the selected sampling rate.
all_records = load_all_ecg_records(
    ptbxl_df,
    sampling_rate=sampling_rate_to_plot,
    data_dir=DATA_DIR,
)

print(f"Loaded ECG records: {len(all_records)}")
print(f"Type of each object in the dictionary: {type(all_records[1])}") # Each value in the dict is a WFDB Record object containing the signal and metadata of one ECG exam

In [ ]:
# Set to None to pick a random record (reproducible via SEED), or set to a specific ecg_id integer to pin a record.
# Example: SELECTED_ECG_ID = 42
SELECTED_ECG_ID = 42

if SELECTED_ECG_ID is None:
    rng = np.random.default_rng(SEED)
    selected_idx = int(rng.integers(0, len(ptbxl_df)))
    selected_row = ptbxl_df.iloc[selected_idx]
else:
    selected_row = ptbxl_df[ptbxl_df["ecg_id"] == SELECTED_ECG_ID].iloc[0]
    selected_idx = ptbxl_df.index[ptbxl_df["ecg_id"] == SELECTED_ECG_ID][0]

selected_patient_id = int(selected_row["patient_id"])
selected_ecg_id = int(selected_row["ecg_id"])
record = all_records[selected_ecg_id]

print(f"Selected row index: {selected_idx}")
print(f"Selected patient_id: {selected_patient_id}")
print(f"Selected ecg_id: {selected_ecg_id}")


In [ ]:
# Plot up to the configured number of seconds of the selected 12-lead ECG.
plot_12_leads_two_columns(
    record,
    sampling_rate=sampling_rate_to_plot,
    patient_id=selected_patient_id,
    ecg_id=selected_ecg_id,
    max_seconds=max_seconds_to_plot,
    apply_standard_normalization=apply_standard_normalization,
)

## Signal Filtering: Notch + Bandpass

Apply a 50 Hz notch filter followed by a 0.5-30 Hz bandpass filter, then visualize one lead across the three stages (raw -> notch -> bandpass).


In [ ]:
def apply_notch_filter(ecg_signal, fs, notch_freq=50.0, quality_factor=30.0):
    """
    ## Description
    Apply a notch filter (default 50 Hz) to attenuate power-line interference.
    
    ## Parameters
    - `ecg_signal`: the input ECG signal to filter (2D array: samples x leads).
    - `fs`: the sampling frequency of the ECG signal (e.g., 100 or 500 Hz for PTB-XL).
    - `notch_freq`: the frequency to notch out (default: 50.0 Hz, common power-line interference frequency).
    - `quality_factor`: the quality factor (Q) of the notch filter, controlling the bandwidth around the notch frequency (default: 30.0, higher means narrower notch).
    """
    # iirnotch returns numerator/denominator coefficients of the notch filter.
    b, a = signal.iirnotch(w0=notch_freq, Q=quality_factor, fs=fs)
    # filtfilt applies forward-backward filtering to avoid phase distortion.
    return signal.filtfilt(b, a, ecg_signal, axis=0)


def apply_bandpass_filter(ecg_signal, fs, lowcut=0.5, highcut=30.0, order=4):
    """
    ## Description
    Apply a Butterworth bandpass filter to retain ECG-relevant frequencies.
    
    ## Parameters
    - `ecg_signal`: the input ECG signal to filter (2D array: samples x leads).
    - `fs`: the sampling frequency of the ECG signal (e.g., 100 or 500 Hz for PTB-XL).
    - `lowcut`: the lower cutoff frequency of the bandpass filter (default: 0.5 Hz).
    - `highcut`: the upper cutoff frequency of the bandpass filter (default: 30.0 Hz).
    - `order`: the order of the Butterworth filter (default: 4).
    """
    nyquist = 0.5 * fs
    # Validate cutoff frequencies before filter design.
    if not (0 < lowcut < highcut < nyquist):
        raise ValueError("Bandpass bounds must satisfy 0 < lowcut < highcut < Nyquist")

    # scipy.signal.butter expects normalized cutoffs in [0, 1] relative to Nyquist.
    normalized_low = lowcut / nyquist
    normalized_high = highcut / nyquist
    b, a = signal.butter(order, [normalized_low, normalized_high], btype="bandpass")
    return signal.filtfilt(b, a, ecg_signal, axis=0)


def plot_filtering_steps(
    raw_signal,
    notch_filtered_signal,
    bandpass_filtered_signal,
    fs,
    lead_index=1,
    max_seconds=10.0,
    lead_name="Lead II",
    apply_standard_normalization=False,
):
    """
    ## Description
    Plot one lead across raw, notch-filtered, and bandpass-filtered stages.
    
    ## Parameters
    - `raw_signal`: the original ECG signal before filtering (2D array: samples x leads).
    - `notch_filtered_signal`: the ECG signal after applying the notch filter (same shape as raw_signal).
    - `bandpass_filtered_signal`: the ECG signal after applying the bandpass filter (same shape as raw_signal).
    - `fs`: the sampling frequency of the signals (e.g., 100 or 500 Hz for PTB-XL).
    - `lead_index`: the index of the lead to plot (default: 1 for Lead II).
    - `max_seconds`: the maximum number of seconds to plot (must be <= 10.0).
    """
    # Keep the same safety bound used in previous plotting sections.
    if max_seconds > 10.0:
        raise ValueError("max_seconds cannot be greater than 10.0")
    if max_seconds <= 0:
        raise ValueError("max_seconds must be greater than 0")

    # Paper-ready font sizes (same style as the 12-lead plotting function).
    title_fs = 13
    label_fs = 12
    tick_fs = 11
    suptitle_fs = 16

    # Restrict all stages to the same visible time window.
    n_samples = min(raw_signal.shape[0], int(max_seconds * fs))
    t = np.arange(n_samples) / fs

    # Copy arrays to avoid in-place edits of original signals during optional normalization.
    stages = [
        ("Raw signal", raw_signal.copy()),
        ("After 50 Hz notch filter", notch_filtered_signal.copy()),
        ("After 0.5-30 Hz bandpass filter", bandpass_filtered_signal.copy()),
    ]

    if apply_standard_normalization:
        # Normalize each stage independently on the displayed window.
        normalized_stages = []
        for stage_name, stage_signal in stages:
            means = np.mean(stage_signal[:n_samples, :], axis=0)
            stds = np.std(stage_signal[:n_samples, :], axis=0)
            stds[stds == 0] = 1.0
            stage_signal[:n_samples, :] = (stage_signal[:n_samples, :] - means) / stds
            normalized_stages.append((stage_name, stage_signal))
        stages = normalized_stages

    # Share y-axis only when standardized, otherwise keep independent scales.
    fig, axes = plt.subplots(
        3,
        1,
        figsize=(16, 12),
        sharex=True,
        sharey=apply_standard_normalization,
    )

    # Plot the chosen lead across the three stages with consistent styling.
    for ax, (stage_name, stage_signal) in zip(axes, stages):
        ax.plot(t, stage_signal[:n_samples, lead_index], linewidth=1.1)
        ax.set_ylabel("z-score" if apply_standard_normalization else "Amplitude [mV]", fontsize=label_fs)
        ax.set_title(f"{stage_name} | {lead_name}", fontsize=title_fs)
        ax.grid(True, alpha=0.7)
        ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)

    axes[-1].set_xlabel("Time [s]", fontsize=label_fs)
    axes[-1].set_xlim(0, max_seconds)

    norm_label = "normalized" if apply_standard_normalization else "raw"
    fig.suptitle(f"Filtering Stages Comparison | {norm_label}", fontsize=suptitle_fs)
    plt.tight_layout()
    save_figure(fig, "filtering_steps")
    plt.show()

In [ ]:
# Use the selected record already loaded in previous cells.
raw_signal = record.p_signal.copy()
fs = sampling_rate_to_plot

# Step 1: remove narrow-band power-line interference at 50 Hz.
# The function expects a 2D array (samples x leads), so we pass the full 12-lead matrix.
signal_after_notch = apply_notch_filter(
    raw_signal,
    fs=fs,
    notch_freq=50.0,
    quality_factor=30.0,
)

# Step 2: keep ECG morphology while reducing baseline/breathing-related low-frequency motion.
signal_after_bandpass = apply_bandpass_filter(
    signal_after_notch,
    fs=fs,
    lowcut=0.5,
    highcut=30.0,
    order=4,
)

ecg_leads = {
    'I': 0,
    'II': 1,
    'III': 2,
    'aVR': 3,
    'aVL': 4,
    'aVF': 5,
    'V1': 6,
    'V2': 7,
    'V3': 8,
    'V4': 9,
    'V5': 10,
    'V6': 11,
}

# Choose one derivation to visualize intermediate stages
lead_index_to_plot = ecg_leads['III']  # Lead II is commonly used for rhythm analysis and has good SNR
lead_name_to_plot = record.sig_name[lead_index_to_plot]

# Compare raw vs notch-filtered vs bandpass-filtered signal on the same time window.
plot_filtering_steps(
    raw_signal,
    signal_after_notch,
    signal_after_bandpass,
    fs=fs,
    lead_index=lead_index_to_plot,
    max_seconds=max_seconds_to_plot,
    lead_name=lead_name_to_plot,
    apply_standard_normalization=apply_standard_normalization,
)

In [ ]:
# Preprocess every loaded ECG record with the same notch + bandpass pipeline.
# The dictionary keeps direct access by ecg_id for later record-level operations.
bandpassed_records = {}

# The list keeps the same row order as ptbxl_df so it can be stacked into a dataset tensor.
bandpassed_signal_list = []
bandpassed_ecg_ids = []

for _, row in ptbxl_df.iterrows():
    ecg_id = int(row["ecg_id"])
    signal_matrix = all_records[ecg_id].p_signal

    signal_notch = apply_notch_filter(
        signal_matrix,
        fs=sampling_rate_to_plot,
        notch_freq=50.0,
        quality_factor=30.0,
    )
    signal_bandpass = apply_bandpass_filter(
        signal_notch,
        fs=sampling_rate_to_plot,
        lowcut=0.5,
        highcut=30.0,
        order=4,
    )

    bandpassed_records[ecg_id] = signal_bandpass
    bandpassed_signal_list.append(signal_bandpass)
    bandpassed_ecg_ids.append(ecg_id)

# Final tensor shape: (n_records, n_samples, n_leads).
bandpassed_signals = np.stack(bandpassed_signal_list, axis=0)
bandpassed_ecg_ids = np.asarray(bandpassed_ecg_ids, dtype=int)

print(f"Bandpassed dataset shape: {bandpassed_signals.shape}")
print(f"Stored records: {len(bandpassed_records)}")


## Frequency-Domain Check

FFT magnitude comparison across the three filtering stages, computed directly on the signal. Dashed lines mark the bandpass upper cutoff (30 Hz) and the power-line frequency (50 Hz).


In [ ]:
def compute_single_sided_fft(signal_1d, fs):
    """
    ## Description
    Return one-sided FFT frequencies and amplitude spectrum for a 1D signal.

    ## Parameters
    - `signal_1d`: the input signal to analyze (1D array).
    - `fs`: the sampling frequency of the signal (e.g., 100 or 500 Hz for PTB-XL).
    """
    n = len(signal_1d)

    window = np.hanning(n)
    windowed = signal_1d * window

    fft_vals = np.fft.rfft(windowed)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)

    # Amplitude scaling for one-sided spectrum (window-corrected).
    scale = 2.0 / np.sum(window)
    amplitude = np.abs(fft_vals) * scale
    return freqs, amplitude


def plot_fft_filtering_stages(
    raw_signal,
    notch_filtered_signal,
    bandpass_filtered_signal,
    fs,
    lead_index=1,
    max_freq_to_show=80.0,
    lead_name="Lead II",
):
    """
    ## Description
    Plot FFT magnitude of one lead across raw, notch-filtered, and bandpass-filtered stages.

    ## Parameters
    - `raw_signal`: the original ECG signal before filtering (2D array: samples x leads).
    - `notch_filtered_signal`: the ECG signal after applying the notch filter (same shape as raw_signal).
    - `bandpass_filtered_signal`: the ECG signal after applying the bandpass filter (same shape as raw_signal).
    - `fs`: the sampling frequency of the signals (e.g., 100 or 500 Hz for PTB-XL).
    - `lead_index`: the index of the lead to analyze (default: 1 for Lead II).
    - `max_freq_to_show`: the maximum frequency to display on the x-axis (default: 80.0 Hz).
    - `lead_name`: the name of the lead to display in the plot titles (default: "Lead II").
    """
    stages = [
        ("Raw signal", raw_signal),
        ("After 50 Hz notch filter", notch_filtered_signal),
        ("After 0.5-30 Hz bandpass filter", bandpass_filtered_signal),
    ]

    title_fs = 13
    label_fs = 12
    tick_fs = 11
    suptitle_fs = 16

    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True, sharey=False)

    for ax, (stage_name, stage_signal) in zip(axes, stages):
        signal_1d = stage_signal[:, lead_index]
        freqs, amp = compute_single_sided_fft(signal_1d, fs)
        ax.plot(freqs, amp, linewidth=1.1)

        ax.set_title(f"{stage_name} | {lead_name}", fontsize=title_fs)
        ax.set_ylabel("Amplitude", fontsize=label_fs)
        ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)
        ax.grid(True, alpha=0.7)
        ax.set_xlim(0.0, min(max_freq_to_show, fs / 2.0))

        # Visual guides for the bandpass upper cutoff and powerline frequency.
        ax.axvline(30.0, color="#34495e", linestyle="--", linewidth=1.2, alpha=0.9)
        ax.axvline(50.0, color="#8e44ad", linestyle="--", linewidth=1.2, alpha=0.9)

    axes[-1].set_xlabel("Frequency [Hz]", fontsize=label_fs)
    axes[-1].set_xlim(-0.5, max_freq_to_show + 0.5)
    fig.suptitle("FFT Comparison Across Filtering Stages", fontsize=suptitle_fs)
    plt.tight_layout()
    save_figure(fig, "fft_filtering_stages")
    plt.show()


In [ ]:
# Reuse the already selected lead from the filtering stage visualization.
# If not available, default to Lead II (index 1).
try:
    lead_idx_fft = lead_index_to_plot
except NameError:
    lead_idx_fft = 1

lead_name_fft = record.sig_name[lead_idx_fft]

plot_fft_filtering_stages(
    raw_signal,
    signal_after_notch,
    signal_after_bandpass,
    fs=fs,
    lead_index=lead_idx_fft,
    max_freq_to_show=55.0,
    lead_name=lead_name_fft,
)

## Standard Normalization (Z-score)

To reduce subject-dependent amplitude/offset differences, each lead is standardized before building the model-ready dataset.

For each signal sample $x$, standard normalization is:

$$
z = \frac{x - \mu}{\sigma}
$$

where:

- $\mu$ is the mean value of the signal.
- $\sigma$ is the standard deviation of the signal.

In this notebook, $\mu$ and $\sigma$ are computed per record and per lead over time:

$$
\mu_{r,l} = \frac{1}{T}\sum_{t=1}^{T} x_{r,t,l}, \qquad
\sigma_{r,l} = \sqrt{\frac{1}{T}\sum_{t=1}^{T} (x_{r,t,l} - \mu_{r,l})^2}
$$

and then applied as:

$$
z_{r,t,l} = \frac{x_{r,t,l} - \mu_{r,l}}{\sigma_{r,l}}
$$

This makes features comparable across subjects while preserving temporal morphology.


In [ ]:
# Standardize each record and lead over time: axis=1 is the time axis in (records, samples, leads).
normalization_means = np.mean(bandpassed_signals, axis=1, keepdims=True)
normalization_stds = np.std(bandpassed_signals, axis=1, keepdims=True)

# Numerical safety: if one lead has zero variance, keep denominator at 1 to avoid division by zero.
normalization_stds[normalization_stds == 0] = 1.0

# Final normalized tensor to be used in the following processing/model-preparation steps.
normalized_bandpassed_signals = (bandpassed_signals - normalization_means) / normalization_stds

# Mirror the same normalized content in a dict keyed by ecg_id for quick record-level access.
normalized_bandpassed_records = {
    int(ecg_id): normalized_bandpassed_signals[i]
    for i, ecg_id in enumerate(bandpassed_ecg_ids)
}

# Quick sanity check: mean close to 0 and std close to 1 for a sample record.
sample_idx = 0
sample_mean = np.mean(normalized_bandpassed_signals[sample_idx], axis=0)
sample_std = np.std(normalized_bandpassed_signals[sample_idx], axis=0)

print(f"Normalized dataset shape: {normalized_bandpassed_signals.shape}")
print(f"Stored normalized records: {len(normalized_bandpassed_records)}")
print(f"Sample record per-lead mean (first 3 leads): {np.round(sample_mean[:3], 4)}")
print(f"Sample record per-lead std  (first 3 leads): {np.round(sample_std[:3], 4)}")


In [ ]:
# Plot one normalized 12-lead record to visually confirm amplitude standardization.
# We reuse the same selected ECG id used in previous sections for easy comparison.
normalized_record = normalized_bandpassed_records[selected_ecg_id]

# Restrict visualization window.
n_samples_norm = min(normalized_record.shape[0], int(max_seconds_to_plot * sampling_rate_to_plot))
t_norm = np.arange(n_samples_norm) / sampling_rate_to_plot
lead_names_norm = record.sig_name

title_fs = 13
label_fs = 12
tick_fs = 11
suptitle_fs = 17

fig, axes = plt.subplots(6, 2, figsize=(16, 16), sharex=True, sharey=True)

for lead_idx in range(12):
    col = 0 if lead_idx < 6 else 1
    row = lead_idx if lead_idx < 6 else lead_idx - 6

    ax = axes[row, col]
    ax.plot(t_norm, normalized_record[:n_samples_norm, lead_idx], linewidth=1.0)
    ax.set_title(lead_names_norm[lead_idx], fontsize=title_fs)
    ax.set_ylabel("z-score", fontsize=label_fs)
    ax.tick_params(axis="both", which="both", length=0, labelsize=tick_fs)
    ax.grid(True, alpha=0.7)

axes[5, 0].set_xlabel("Time [s]", fontsize=label_fs)
axes[5, 1].set_xlabel("Time [s]", fontsize=label_fs)
axes[0, 0].set_xlim(0, max_seconds_to_plot)

fig.suptitle(
    f"Normalized ECG (Z-score) | patient_id={selected_patient_id} | ecg_id={selected_ecg_id}",
    fontsize=suptitle_fs,
)
plt.tight_layout()
save_figure(fig, "normalized_12_leads")
plt.show()


## Healthy vs Non-healthy Labeling From Metadata

In this section, we assign a clinical class to each ECG using PTB-XL metadata.

Workflow used:

1. Load SCP statements metadata (`scp_statements.csv`) and keep diagnostic codes.
2. Parse the `scp_codes` field of each ECG in `ptbxl_database.csv`.
3. Mark an ECG as **healthy** only when its diagnostic code set is exactly `{NORM}`.
4. Aggregate at patient level: a patient is **healthy** only if all their available ECGs are healthy.
5. Select one healthy and one non-healthy example to compare their processed and normalized signals.

This gives a reproducible and clinically grounded way to define labels before model dataset construction.


In [ ]:
# Build healthy/non-healthy labels from PTB-XL metadata using diagnostic SCP codes.
scp_path = f"{DATA_DIR}/scp_statements.csv"
scp_df = pd.read_csv(scp_path, index_col=0)
diagnostic_codes = set(scp_df.index[scp_df["diagnostic"] == 1])


def parse_scp_codes_field(raw_scp_codes):
    if isinstance(raw_scp_codes, dict):
        return raw_scp_codes
    if pd.isna(raw_scp_codes):
        return {}
    return ast.literal_eval(raw_scp_codes)


def is_healthy_record_strict(raw_scp_codes):
    parsed = parse_scp_codes_field(raw_scp_codes)
    record_codes = set(parsed.keys())
    diagnostic_in_record = record_codes.intersection(diagnostic_codes)
    return diagnostic_in_record == {"NORM"}


# Record-level label.
ptbxl_labeled_df = ptbxl_df.copy()
ptbxl_labeled_df["is_healthy_record"] = ptbxl_labeled_df["scp_codes"].apply(is_healthy_record_strict)

# Patient-level label: healthy only if all available records are healthy.
patient_labels_df = (
    ptbxl_labeled_df.groupby("patient_id", as_index=False)
    .agg(
        is_healthy_patient=("is_healthy_record", "all"),
        n_records=("ecg_id", "count"),
    )
)

healthy_patients = patient_labels_df.loc[patient_labels_df["is_healthy_patient"], "patient_id"].tolist()
nonhealthy_patients = patient_labels_df.loc[~patient_labels_df["is_healthy_patient"], "patient_id"].tolist()

if len(healthy_patients) == 0 or len(nonhealthy_patients) == 0:
    raise RuntimeError("Could not find both healthy and non-healthy patients in the current subset.")

# Pick one representative ECG per class, ensuring it is available in the normalized dictionary.
healthy_row = ptbxl_labeled_df[
    (ptbxl_labeled_df["patient_id"] == healthy_patients[0])
    & (ptbxl_labeled_df["ecg_id"].isin(normalized_bandpassed_records.keys()))
].iloc[0]

nonhealthy_row = ptbxl_labeled_df[
    (ptbxl_labeled_df["patient_id"] == nonhealthy_patients[0])
    & (ptbxl_labeled_df["ecg_id"].isin(normalized_bandpassed_records.keys()))
].iloc[0]

healthy_patient_id = int(healthy_row["patient_id"])
healthy_ecg_id = int(healthy_row["ecg_id"])
nonhealthy_patient_id = int(nonhealthy_row["patient_id"])
nonhealthy_ecg_id = int(nonhealthy_row["ecg_id"])

print(f"Healthy patient selected: patient_id={healthy_patient_id}, ecg_id={healthy_ecg_id}")
print(f"Non-healthy patient selected: patient_id={nonhealthy_patient_id}, ecg_id={nonhealthy_ecg_id}")
print(patient_labels_df["is_healthy_patient"].value_counts().rename(index={True: 'Healthy', False: 'Non-healthy'}))


In [ ]:
# Full 12-lead comparison as two separate figures (healthy and non-healthy).
healthy_signal_12 = normalized_bandpassed_records[healthy_ecg_id]
nonhealthy_signal_12 = normalized_bandpassed_records[nonhealthy_ecg_id]

n_samples_compare_12 = min(
    healthy_signal_12.shape[0],
    nonhealthy_signal_12.shape[0],
    int(max_seconds_to_plot * sampling_rate_to_plot),
)
t_compare_12 = np.arange(n_samples_compare_12) / sampling_rate_to_plot
lead_names_12 = all_records[healthy_ecg_id].sig_name

# Extract pathology text for the selected non-healthy ECG from SCP diagnostic codes.
nonhealthy_codes_dict = parse_scp_codes_field(nonhealthy_row["scp_codes"])
nonhealthy_diagnostic_codes = sorted(
    set(nonhealthy_codes_dict.keys()).intersection(diagnostic_codes).difference({"NORM"})
)

if len(nonhealthy_diagnostic_codes) == 0:
    nonhealthy_pathology_text = "No diagnostic pathology code available"
else:
    pathology_entries = []
    for code in nonhealthy_diagnostic_codes:
        if code in scp_df.index:
            pathology_entries.append(f"{code}: {scp_df.loc[code, 'description']}")
        else:
            pathology_entries.append(code)
    nonhealthy_pathology_text = "; ".join(pathology_entries)


def plot_single_subject_12_leads(signal_matrix, line_color, suptitle, output_suffix):
    fig, axes = plt.subplots(6, 2, figsize=(16, 16), sharex=True, sharey=True)

    for lead_idx in range(12):
        col = 0 if lead_idx < 6 else 1
        row = lead_idx if lead_idx < 6 else lead_idx - 6

        ax = axes[row, col]
        ax.plot(
            t_compare_12,
            signal_matrix[:n_samples_compare_12, lead_idx],
            color=line_color,
            linewidth=1.0,
        )
        ax.set_title(lead_names_12[lead_idx], fontsize=12)
        ax.set_ylabel("z-score", fontsize=11)
        ax.grid(True, alpha=0.7)
        ax.tick_params(axis="both", which="both", length=0, labelsize=10)

    axes[5, 0].set_xlabel("Time [s]", fontsize=11)
    axes[5, 1].set_xlabel("Time [s]", fontsize=11)
    axes[0, 0].set_xlim(0, max_seconds_to_plot)

    fig.suptitle(suptitle, fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    save_figure(fig, output_suffix)
    plt.show()


plot_single_subject_12_leads(
    healthy_signal_12,
    line_color="#2e8b57",
    suptitle=f"Processed + Normalized ECG (Healthy) | patient_id={healthy_patient_id} | ecg_id={healthy_ecg_id}",
    output_suffix="normalized_healthy_12leads",
)

plot_single_subject_12_leads(
    nonhealthy_signal_12,
    line_color="#c0392b",
    suptitle=(
        f"Processed + Normalized ECG (Non-healthy) | patient_id={nonhealthy_patient_id} | ecg_id={nonhealthy_ecg_id}\n"
        f"Pathology: {nonhealthy_pathology_text}"
    ),
    output_suffix="normalized_nonhealthy_12leads",
)

print(f"Non-healthy pathology: {nonhealthy_pathology_text}")


In [ ]:
# Plot processed+normalized signals for one healthy and one non-healthy patient.
# Lead II is a common choice for rhythm/morphology inspection.
lead_name_to_compare = "II"
lead_idx_compare = ecg_leads[lead_name_to_compare]

healthy_signal = normalized_bandpassed_records[healthy_ecg_id]
nonhealthy_signal = normalized_bandpassed_records[nonhealthy_ecg_id]

n_samples_compare = min(
    healthy_signal.shape[0],
    nonhealthy_signal.shape[0],
    int(max_seconds_to_plot * sampling_rate_to_plot),
)
t_compare = np.arange(n_samples_compare) / sampling_rate_to_plot

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True, sharey=True)

axes[0].plot(t_compare, healthy_signal[:n_samples_compare, lead_idx_compare], color="#2e8b57", linewidth=1.1)
axes[0].set_title(
    f"Healthy | patient_id={healthy_patient_id} | ecg_id={healthy_ecg_id} | lead {lead_name_to_compare}",
    fontsize=12,
)
axes[0].set_ylabel("z-score", fontsize=11)
axes[0].grid(True, alpha=0.7)
axes[0].tick_params(axis="both", which="both", length=0, labelsize=10)

axes[1].plot(t_compare, nonhealthy_signal[:n_samples_compare, lead_idx_compare], color="#c0392b", linewidth=1.1)
axes[1].set_title(
    f"Non-healthy | patient_id={nonhealthy_patient_id} | ecg_id={nonhealthy_ecg_id} | lead {lead_name_to_compare}",
    fontsize=12,
)
axes[1].set_ylabel("z-score", fontsize=11)
axes[1].set_xlabel("Time [s]", fontsize=11)
axes[1].grid(True, alpha=0.7)
axes[1].tick_params(axis="both", which="both", length=0, labelsize=10)

axes[0].set_xlim(0, max_seconds_to_plot)
fig.suptitle("Processed + Normalized ECG Comparison", fontsize=14)
plt.tight_layout()
save_figure(fig, "normalized_healthy_vs_nonhealthy_leadII")
plt.show()


## Time-Domain and Frequency-Domain HRV Features

Before building the model dataset, we extract interpretable HRV features from **R-R intervals** (time between consecutive R peaks), detected from **Lead II** of the preprocessed normalized ECG.

### Time-domain features (per record)

| Feature | Description | Formula | Unit |
|---|---|---|---|
| Mean Heart Rate | Average beats per minute derived from mean R-R interval | $\overline{\text{HR}} = \dfrac{60}{\overline{\text{RR}}}$ | bpm |
| SDNN | Standard deviation of R-R intervals (overall variability) | $\text{SDNN} = \sqrt{\dfrac{1}{N-1}\sum_{i=1}^{N}(\text{RR}_i-\overline{\text{RR}})^2}$ | ms |
| RMSSD | Root mean square of successive R-R differences (short-term variability) | $\text{RMSSD} = \sqrt{\dfrac{1}{N-1}\sum_{i=1}^{N-1}(\text{RR}_{i+1}-\text{RR}_i)^2}$ | ms |
| pNN50 | Percentage of successive R-R interval differences greater than 50 ms | $\text{pNN50}=100\cdot\dfrac{\text{NN50}}{N-1}$ | % |

where $\text{NN50}$ is the number of successive interval pairs such that $|\Delta \text{RR}| > 50\,\text{ms}$.

### Frequency-domain features (per record)

After interpolating the RR tachogram, we estimate PSD with Welch and integrate power in standard bands:

- **LF power**: 0.04-0.15 Hz
- **HF power**: 0.15-0.40 Hz
- **LF/HF ratio**: $\dfrac{\text{LF}}{\text{HF}}$

These metrics are commonly used to assess **autonomic nervous system (ANS)** regulation and sympathovagal balance.

All features are computed per record, then averaged at patient level.


In [ ]:
# Lead II is the standard clinical choice for R-peak detection (good P-QRS-T morphology and SNR).
LEAD_HRV = ecg_leads["II"]


def compute_rr_intervals(signal_1d, fs):
    """
    Detect R peaks with NeuroKit2 and return:
    - rr_sec: R-R intervals in seconds
    - r_peaks: R peak sample indices
    Returns (None, None) if too few peaks are detected.
    """
    cleaned = nk.ecg_clean(signal_1d, sampling_rate=fs)
    _, info = nk.ecg_peaks(cleaned, sampling_rate=fs)
    r_peaks = info["ECG_R_Peaks"]
    if len(r_peaks) < 3:
        return None, None
    rr_sec = np.diff(r_peaks) / fs
    return rr_sec, r_peaks


def compute_hrv_time_features(rr_sec):
    """
    Compute time-domain HRV features from R-R intervals (in seconds).

    Returns
    -------
    mean_hr_bpm : float
    sdnn_ms     : float
    rmssd_ms    : float
    pnn50_pct   : float
    """
    mean_rr = np.mean(rr_sec)
    mean_hr_bpm = 60.0 / mean_rr

    rr_ms = rr_sec * 1000.0
    sdnn_ms = np.std(rr_ms, ddof=1)

    successive_diffs_ms = np.diff(rr_ms)
    rmssd_ms = np.sqrt(np.mean(successive_diffs_ms ** 2))

    if len(successive_diffs_ms) == 0:
        pnn50_pct = np.nan
    else:
        pnn50_pct = 100.0 * np.mean(np.abs(successive_diffs_ms) > 50.0)

    return mean_hr_bpm, sdnn_ms, rmssd_ms, pnn50_pct


def compute_hrv_freq_features(rr_sec, r_peaks, fs, interp_fs=4.0):
    """
    Compute LF, HF, and LF/HF from RR intervals.

    Steps:
    1) Build beat times from detected R peaks.
    2) Interpolate RR series on an evenly sampled grid.
    3) Estimate PSD via Welch and integrate LF/HF bands.
    """
    if len(rr_sec) < 4 or r_peaks is None:
        return np.nan, np.nan, np.nan

    rr_ms = rr_sec * 1000.0

    # RR interval i is associated with the second beat time of the pair.
    rr_times_sec = r_peaks[1:] / fs

    if len(rr_times_sec) < 4:
        return np.nan, np.nan, np.nan

    t_min, t_max = rr_times_sec[0], rr_times_sec[-1]
    if t_max <= t_min:
        return np.nan, np.nan, np.nan

    t_uniform = np.arange(t_min, t_max, 1.0 / interp_fs)
    if len(t_uniform) < 8:
        return np.nan, np.nan, np.nan

    rr_uniform_ms = np.interp(t_uniform, rr_times_sec, rr_ms)
    rr_uniform_ms = signal.detrend(rr_uniform_ms)

    nperseg = min(256, len(rr_uniform_ms))
    if nperseg < 8:
        return np.nan, np.nan, np.nan

    freqs, psd = signal.welch(rr_uniform_ms, fs=interp_fs, nperseg=nperseg)

    lf_mask = (freqs >= 0.04) & (freqs < 0.15)
    hf_mask = (freqs >= 0.15) & (freqs <= 0.40)

    lf_power = np.trapezoid(psd[lf_mask], freqs[lf_mask]) if np.any(lf_mask) else np.nan
    hf_power = np.trapezoid(psd[hf_mask], freqs[hf_mask]) if np.any(hf_mask) else np.nan

    if np.isnan(lf_power) or np.isnan(hf_power) or hf_power <= 0:
        lf_hf_ratio = np.nan
    else:
        lf_hf_ratio = lf_power / hf_power

    return lf_power, hf_power, lf_hf_ratio


# ------------------------------------------------------------------
# Compute HRV features for every record in the normalized dataset.
# ------------------------------------------------------------------
records_hrv = []

for ecg_id, signal_matrix in normalized_bandpassed_records.items():
    signal_1d = signal_matrix[:, LEAD_HRV]
    rr_intervals, r_peaks = compute_rr_intervals(signal_1d, fs=sampling_rate_to_plot)

    if rr_intervals is None:
        print(f"  [WARN] ecg_id={ecg_id}: insufficient R peaks detected, skipping.")
        continue

    mean_hr_bpm, sdnn_ms, rmssd_ms, pnn50_pct = compute_hrv_time_features(rr_intervals)
    lf_power, hf_power, lf_hf_ratio = compute_hrv_freq_features(
        rr_intervals,
        r_peaks,
        fs=sampling_rate_to_plot,
        interp_fs=4.0,
    )

    patient_id = int(
        ptbxl_labeled_df.loc[ptbxl_labeled_df["ecg_id"] == ecg_id, "patient_id"].iloc[0]
    )

    records_hrv.append(
        {
            "ecg_id": ecg_id,
            "patient_id": patient_id,
            "mean_hr_bpm": mean_hr_bpm,
            "sdnn_ms": sdnn_ms,
            "rmssd_ms": rmssd_ms,
            "pnn50_pct": pnn50_pct,
            "lf_power": lf_power,
            "hf_power": hf_power,
            "lf_hf_ratio": lf_hf_ratio,
        }
    )

hrv_records_df = pd.DataFrame(records_hrv)

# ------------------------------------------------------------------
# Aggregate at patient level: average feature values across records.
# ------------------------------------------------------------------
patient_hrv_df = (
    hrv_records_df.groupby("patient_id", as_index=False)
    .agg(
        mean_hr_bpm=("mean_hr_bpm", "mean"),
        sdnn_ms=("sdnn_ms", "mean"),
        rmssd_ms=("rmssd_ms", "mean"),
        pnn50_pct=("pnn50_pct", "mean"),
        lf_power=("lf_power", "mean"),
        hf_power=("hf_power", "mean"),
        lf_hf_ratio=("lf_hf_ratio", "mean"),
    )
)

# Attach the health label so features and class are always aligned.
patient_hrv_df = patient_hrv_df.merge(
    patient_labels_df[["patient_id", "is_healthy_patient"]],
    on="patient_id",
    how="left",
)

feature_cols = [
    "mean_hr_bpm",
    "sdnn_ms",
    "rmssd_ms",
    "pnn50_pct",
    "lf_power",
    "hf_power",
    "lf_hf_ratio",
]

print(f"HRV features computed for {len(hrv_records_df)} records / {len(patient_hrv_df)} patients")
print(f"\nPer-class summary (mean ± std):")
display(
    patient_hrv_df.groupby("is_healthy_patient")[feature_cols]
    .agg(["mean", "std"])
    .round(2)
    .rename(index={True: "Healthy", False: "Non-healthy"})
)

## Train / Test Split on HRV Metrics

Now we split the dataset using **HRV metrics**.

As done previously, the split is **patient-level stratified**:

- patients are separated by class (healthy / non-healthy),
- each class is shuffled with a fixed seed,
- each class is split with the same test ratio.

This preserves class proportions and prevents leakage by construction (one patient appears in only one split).

Output arrays:

- `X_hrv_train`, `X_hrv_test`
- `y_hrv_train`, `y_hrv_test`

Label convention used here:

- `0` = healthy
- `1` = unhealthy


In [ ]:
# Patient-level stratified train/test split using HRV metrics.
TEST_RATIO_HRV = 1 / 5
rng_split_hrv = np.random.default_rng(SEED)

healthy_pids = sorted(
    patient_hrv_df.loc[patient_hrv_df["is_healthy_patient"], "patient_id"].tolist()
)
nonhealthy_pids = sorted(
    patient_hrv_df.loc[~patient_hrv_df["is_healthy_patient"], "patient_id"].tolist()
)

rng_split_hrv.shuffle(healthy_pids)
rng_split_hrv.shuffle(nonhealthy_pids)

n_healthy_test = max(1, round(len(healthy_pids) * TEST_RATIO_HRV))
n_nonhealthy_test = max(1, round(len(nonhealthy_pids) * TEST_RATIO_HRV))

test_pids_hrv = set(healthy_pids[:n_healthy_test] + nonhealthy_pids[:n_nonhealthy_test])
train_pids_hrv = set(healthy_pids[n_healthy_test:] + nonhealthy_pids[n_nonhealthy_test:])

HRV_FEATURE_COLS = [
    "mean_hr_bpm",
    "sdnn_ms",
    "rmssd_ms",
    "pnn50_pct",
    "lf_power",
    "hf_power",
    "lf_hf_ratio",
]

hrv_train_df = patient_hrv_df[patient_hrv_df["patient_id"].isin(train_pids_hrv)].reset_index(drop=True)
hrv_test_df = patient_hrv_df[patient_hrv_df["patient_id"].isin(test_pids_hrv)].reset_index(drop=True)

X_hrv_train = hrv_train_df[HRV_FEATURE_COLS].to_numpy(dtype=np.float32)
X_hrv_test = hrv_test_df[HRV_FEATURE_COLS].to_numpy(dtype=np.float32)

# Original label in patient_hrv_df: 1=healthy, 0=non-healthy.
# Requested convention here: 0=healthy, 1=unhealthy.
y_hrv_train = 1 - hrv_train_df["is_healthy_patient"].astype(int).to_numpy(dtype=np.int32)
y_hrv_test = 1 - hrv_test_df["is_healthy_patient"].astype(int).to_numpy(dtype=np.int32)

print(f"TEST_RATIO_HRV: {TEST_RATIO_HRV:.2f}")
print(f"Patients -> train: {len(train_pids_hrv)} | test: {len(test_pids_hrv)}")
print(f"X_hrv_train shape: {X_hrv_train.shape}")
print(f"X_hrv_test  shape: {X_hrv_test.shape}")
print()
print("Class distribution (0=healthy, 1=unhealthy):")
for split_name, y_split in [("Train", y_hrv_train), ("Test", y_hrv_test)]:
    n_total = len(y_split)
    n_healthy = int(np.sum(y_split == 0))
    n_unhealthy = int(np.sum(y_split == 1))
    print(
        f"{split_name:5s} | {n_total:3d} patients | "
        f"Healthy: {n_healthy} ({100*n_healthy/n_total:.1f}%) | "
        f"Unhealthy: {n_unhealthy} ({100*n_unhealthy/n_total:.1f}%)"
    )


## Train Classifiers on HRV Metrics

We now train a set of classifiers on `X_hrv_train, y_hrv_train`, evaluate them on the test set, and select the best model based on **test balanced accuracy**.

Then we plot the **confusion matrix** of the selected best model.

### Clinical Ranking Metrics (Positive class = Unhealthy = 1)

In this section, models are ranked primarily by unhealthy recall (sensitivity).

Let TP, TN, FP, FN be computed with class 1 = unhealthy as positive.

$$
\text{Recall}_{\text{unhealthy}} = \frac{TP}{TP + FN}
$$

$$
\text{Precision}_{\text{unhealthy}} = \frac{TP}{TP + FP}
$$

$$
\text{Specificity}_{\text{healthy}} = \frac{TN}{TN + FP}
$$

$$
F_{\beta} = (1 + \beta^2)\cdot\frac{\text{Precision}\cdot\text{Recall}}{\beta^2\cdot\text{Precision} + \text{Recall}}, \quad \beta=2
$$

$$
\text{Balanced Accuracy} = \frac{\text{Recall}_{\text{unhealthy}} + \text{Specificity}_{\text{healthy}}}{2}
$$

Ranking policy used below: Recall_unhealthy first, then F2, then balanced accuracy and specificity.

In [ ]:
# Shared utilities for clinical model ranking and paper-ready confusion-matrix plots.
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

import seaborn as sns

CLINICAL_RANKING_KEYS = [
    "test_recall_unhealthy",
    "test_f2_unhealthy",
    "test_balanced_accuracy",
    "test_specificity_healthy",
    "test_precision_unhealthy",
]


def ensure_split_vars_are_available():
    required_vars = ["X_hrv_train", "X_hrv_test", "y_hrv_train", "y_hrv_test"]
    missing_vars = [v for v in required_vars if v not in globals()]
    if missing_vars:
        raise RuntimeError(
            "Missing variables for training/tuning: " + ", ".join(missing_vars) + ". "
            "Run the HRV train/test split cell first."
        )


def compute_clinical_metrics(y_true, y_pred):
    rec_unhealthy = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    prec_unhealthy = precision_score(y_true, y_pred, pos_label=1, zero_division=0)
    spec_healthy = recall_score(y_true, y_pred, pos_label=0, zero_division=0)
    f2_unhealthy = fbeta_score(y_true, y_pred, beta=2, pos_label=1, zero_division=0)

    return {
        "test_recall_unhealthy": rec_unhealthy,
        "test_precision_unhealthy": prec_unhealthy,
        "test_specificity_healthy": spec_healthy,
        "test_f2_unhealthy": f2_unhealthy,
        "test_balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "test_accuracy": accuracy_score(y_true, y_pred),
        "test_f1": f1_score(y_true, y_pred, zero_division=0),
    }


def plot_confusion_matrix_paper(y_true, y_pred, title, output_suffix):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_row_norm = np.divide(
        cm.astype(float),
        row_sums,
        out=np.zeros_like(cm, dtype=float),
        where=row_sums != 0,
    )

    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f"{cm[i, j]}\n({cm_row_norm[i, j]*100:.1f}%)"

    sns.set_theme(style="white", context="paper")
    fig, ax = plt.subplots(figsize=(6.8, 5.8), dpi=300)
    sns.heatmap(
        cm_row_norm,
        annot=annot,
        fmt="",
        cmap="Blues",
        cbar=True,
        cbar_kws={"label": "Row-normalized proportion"},
        vmin=0.0,
        vmax=1.0,
        linewidths=0.8,
        linecolor="white",
        square=True,
        xticklabels=["Healthy (0)", "Unhealthy (1)"],
        yticklabels=["Healthy (0)", "Unhealthy (1)"],
        annot_kws={"fontsize": 10},
        ax=ax,
    )
    ax.set_xlabel("Predicted label", fontsize=11)
    ax.set_ylabel("True label", fontsize=11)
    ax.set_title(title, fontsize=12, pad=10)
    ax.tick_params(axis="both", labelsize=10)

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=9)
    cbar.set_label("Row-normalized proportion", fontsize=10)

    plt.tight_layout()
    save_figure(fig, output_suffix)
    plt.show()

In [ ]:
# Train and compare baseline classifiers on HRV metrics, then plot confusion matrix of the best.
ensure_split_vars_are_available()

models = {
    "Logistic Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, random_state=SEED)),
    ]),
    "SVM (RBF)": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED)),
    ]),
    "KNN": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5)),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            random_state=SEED,
            class_weight="balanced",
        )),
    ]),
    "Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", GradientBoostingClassifier(random_state=SEED)),
    ]),
}

results = []
reports = {}

for model_name, model in models.items():
    model.fit(X_hrv_train, y_hrv_train)
    y_pred = model.predict(X_hrv_test)

    metric_row = {"model": model_name}
    metric_row.update(compute_clinical_metrics(y_hrv_test, y_pred))
    results.append(metric_row)
    reports[model_name] = {"model": model, "y_pred": y_pred}

results_df = pd.DataFrame(results).sort_values(
    by=CLINICAL_RANKING_KEYS,
    ascending=False,
).reset_index(drop=True)

print("Model ranking on test set (clinical priority: unhealthy recall first):")
display(
    results_df[[
        "model",
        "test_recall_unhealthy",
        "test_precision_unhealthy",
        "test_specificity_healthy",
        "test_f2_unhealthy",
        "test_balanced_accuracy",
        "test_accuracy",
    ]].round(3)
)

best_model_name = results_df.loc[0, "model"]
best_model = reports[best_model_name]["model"]
y_test_pred = reports[best_model_name]["y_pred"]

print(f"Best model (clinical ranking): {best_model_name}")
plot_confusion_matrix_paper(
    y_hrv_test,
    y_test_pred,
    title=f"Test Confusion Matrix - {best_model_name}",
    output_suffix="best_model_confusion_matrix_hrv_metrics",
)

## Hyperparameter Tuning and Best Tuned Model Confusion Matrix

In this final section, we tune model hyperparameters with cross-validation on the training split only and then evaluate each tuned model on the test set.

Clinical ranking policy remains the same: unhealthy recall first, then F2, then balanced accuracy and specificity.

In [ ]:
# Hyperparameter tuning with CV on train split, then test evaluation of tuned models.
ensure_split_vars_are_available()

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

tuning_configs = {
    "Logistic Regression": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=4000, random_state=SEED)),
        ]),
        "param_grid": {
            "clf__C": [0.1, 1.0, 10.0],
            "clf__solver": ["lbfgs"],
        },
    },
    "SVM (RBF)": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", random_state=SEED)),
        ]),
        "param_grid": {
            "clf__C": [0.5, 1.0, 5.0, 10.0],
            "clf__gamma": ["scale", 0.1, 0.01],
        },
    },
    "KNN": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier()),
        ]),
        "param_grid": {
            "clf__n_neighbors": [3, 5, 7, 9],
            "clf__weights": ["uniform", "distance"],
            "clf__p": [1, 2],
        },
    },
    "Random Forest": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(random_state=SEED, class_weight="balanced")),
        ]),
        "param_grid": {
            "clf__n_estimators": [200, 400],
            "clf__max_depth": [None, 8, 16],
            "clf__min_samples_split": [2, 5],
            "clf__min_samples_leaf": [1, 2],
        },
    },
    "Gradient Boosting": {
        "pipeline": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", GradientBoostingClassifier(random_state=SEED)),
        ]),
        "param_grid": {
            "clf__n_estimators": [100, 200],
            "clf__learning_rate": [0.03, 0.1],
            "clf__max_depth": [2, 3],
            "clf__subsample": [0.8, 1.0],
        },
    },
}

tuned_results = []
tuned_reports = {}

for model_name, cfg in tuning_configs.items():
    print(f"\nTuning {model_name}...")
    grid = GridSearchCV(
        estimator=cfg["pipeline"],
        param_grid=cfg["param_grid"],
        scoring="recall",
        cv=cv_strategy,
        n_jobs=-1,
        refit=True,
    )
    grid.fit(X_hrv_train, y_hrv_train)

    best_estimator = grid.best_estimator_
    y_pred_test = best_estimator.predict(X_hrv_test)

    metric_row = {
        "model": model_name,
        "cv_best_recall_unhealthy": grid.best_score_,
    }
    metric_row.update(compute_clinical_metrics(y_hrv_test, y_pred_test))
    metric_row["best_params"] = grid.best_params_
    tuned_results.append(metric_row)

    tuned_reports[model_name] = {
        "best_estimator": best_estimator,
        "y_pred_test": y_pred_test,
    }

tuned_results_df = pd.DataFrame(tuned_results).sort_values(
    by=CLINICAL_RANKING_KEYS,
    ascending=False,
).reset_index(drop=True)

print("\nTuned model ranking on test set (clinical priority: unhealthy recall first):")
display(tuned_results_df[[
    "model",
    "cv_best_recall_unhealthy",
    "test_recall_unhealthy",
    "test_precision_unhealthy",
    "test_specificity_healthy",
    "test_f2_unhealthy",
    "test_balanced_accuracy",
    "test_accuracy",
]].round(3))

best_tuned_model_name = tuned_results_df.loc[0, "model"]
best_tuned_model = tuned_reports[best_tuned_model_name]["best_estimator"]
y_test_pred_tuned = tuned_reports[best_tuned_model_name]["y_pred_test"]

print(f"Best tuned model (clinical ranking): {best_tuned_model_name}")
print("Best hyperparameters:")
print(tuned_results_df.loc[0, "best_params"])

plot_confusion_matrix_paper(
    y_hrv_test,
    y_test_pred_tuned,
    title=f"Tuned Model Confusion Matrix - {best_tuned_model_name}",
    output_suffix="best_tuned_model_confusion_matrix_hrv_metrics",
)